In [0]:
# from pyspark.sql.functions import col, trim

# bronze = spark.table("notebook_catalog.notebook_schema.sales_bronze")

# silver = (bronze
#     .select(
#         col("order_id").cast("int").alias("order_id"),
#         trim(col("product")).alias("product"),
#         col("unit_price").cast("double").alias("unit_price"))
#     .filter(col("order_id").isNotNull() & col("product").isNotNull() & col("unit_price").isNotNull())
#     .dropDuplicates(["order_id"]))

# silver.write.mode("overwrite").saveAsTable("notebook_catalog.notebook_schema.sales_silver")   # <- silver table

In [0]:
# display(silver)

In [0]:
# from pyspark.sql.functions import col, trim
# from delta.tables import DeltaTable

# # 1. Clean the bronze data
# cleaned = (spark.table("notebook_catalog.notebook_schema.sales_bronze")
#     .select(
#         col("order_id").cast("int").alias("order_id"),
#         trim(col("product")).alias("product"),
#         col("unit_price").cast("double").alias("unit_price"))
#     .filter(col("order_id").isNotNull() & col("unit_price").isNotNull())
#     .dropDuplicates(["order_id"]))

# # 2. Create the silver table once if it doesn't exist (empty, same columns)
# cleaned.limit(0).write.mode("ignore").saveAsTable("notebook_catalog.notebook_schema.sales_silver")

# # 3. Merge
# silver = DeltaTable.forName(spark, "notebook_catalog.notebook_schema.sales_silver")

# (silver.alias("t")
#     .merge(cleaned.alias("s"), "t.order_id = s.order_id")
#     .whenMatchedUpdateAll()
#     .whenNotMatchedInsertAll()
#     .execute())

In [0]:
# %sql
# SELECT order_id, COUNT(*) FROM notebook_catalog.notebook_schema.sales_silver
# GROUP BY order_id HAVING COUNT(*) > 1

In [0]:
# display(cleaned)

In [0]:
%sql
DROP TABLE IF EXISTS notebook_catalog.notebook_schema.sales_silver;
DROP TABLE IF EXISTS notebook_catalog.notebook_schema.sales_gold;

In [0]:
from pyspark.sql.functions import col, trim, when, lit, current_timestamp
from delta.tables import DeltaTable

cat = "notebook_catalog.notebook_schema"

# 1. Read bronze, keep raw text copies, and convert types
typed = (spark.table(f"{cat}.sales_bronze")
    .select(
        col("order_id").alias("raw_order_id"),
        col("quantity").alias("raw_quantity"),
        col("unit_price").alias("raw_unit_price"),
        col("order_id").cast("int").alias("order_id"),
        trim(col("product")).alias("product"),
        col("quantity").cast("int").alias("quantity"),
        col("unit_price").cast("double").alias("unit_price")))

# 2. Label each row with a failure reason (null = good row)
checked = typed.withColumn(
    "reason",
    when(col("order_id").isNull(), lit("order_id missing or not a number"))
    .when(col("product").isNull() | (col("product") == ""), lit("product is empty"))
    .when(col("quantity").isNull(), lit("quantity missing or not a number"))
    .when(col("quantity") <= 0, lit("quantity must be greater than 0"))
    .when(col("unit_price").isNull(), lit("unit_price missing or not a number"))
    .when(col("unit_price") <= 0, lit("unit_price must be greater than 0"))
)

# 3. Split into good and bad
good = (checked.filter(col("reason").isNull())
        .select("order_id", "product", "quantity", "unit_price")
        .withColumn("total_amount", col("quantity") * col("unit_price"))
        .dropDuplicates(["order_id"]))

bad = (checked.filter(col("reason").isNotNull())
       .select("raw_order_id", "product", "raw_quantity", "raw_unit_price", "reason")
       .withColumn("quarantined_at", current_timestamp()))

# 4. Save bad rows to the quarantine table
bad.write.mode("overwrite").saveAsTable(f"{cat}.sales_quarantine")

# 5. Merge good rows into silver
good.limit(0).write.mode("ignore").saveAsTable(f"{cat}.sales_silver")
silver = DeltaTable.forName(spark, f"{cat}.sales_silver")

(silver.alias("t")
    .merge(good.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())